In [5]:
import random
import statistics

# ==========================================
# 1. DADOS DO CENÁRIO (15 Microsserviços)
# ==========================================
VALUES = [45, 25, 35, 60, 20, 50, 40, 30, 55, 65, 15, 25, 50, 45, 35]
RAM    = [2.0, 1.0, 1.5, 3.0, 0.5, 2.5, 2.0, 1.0, 2.5, 3.5, 0.5, 1.0, 2.0, 1.5, 1.0] # Limite: 16 GB
CPU    = [1.0, 0.5, 0.8, 1.5, 0.2, 1.2, 1.0, 0.5, 1.2, 1.8, 0.2, 0.5, 1.0, 0.8, 0.5] # Limite: 8 Cores

MAX_RAM = 16.0
MAX_CPU = 8.0

# ==========================================
# 2. FUNÇÕES DE AVALIAÇÃO DE FITNESS
# ==========================================
def evaluate_strategy_a(individual):
    """Estratégia A: Penalidade Rígida (Violação = 0)"""
    total_val = sum(v * gene for v, gene in zip(VALUES, individual))
    total_ram = sum(r * gene for r, gene in zip(RAM, individual))
    total_cpu = sum(c * gene for c, gene in zip(CPU, individual))

    if total_ram > MAX_RAM or total_cpu > MAX_CPU:
        return 0.0
    return float(total_val)

def evaluate_strategy_b(individual):
    """Estratégia B: Penalidade Proporcional"""
    total_val = sum(v * gene for v, gene in zip(VALUES, individual))
    total_ram = sum(r * gene for r, gene in zip(RAM, individual))
    total_cpu = sum(c * gene for c, gene in zip(CPU, individual))

    penalty = 0.0
    if total_ram > MAX_RAM:
        penalty += (total_ram - MAX_RAM) * 30.0
    if total_cpu > MAX_CPU:
        penalty += (total_cpu - MAX_CPU) * 30.0

    fitness = total_val - penalty
    return max(0.0, fitness)

# ==========================================
# 3. OPERADORES GENÉTICOS
# ==========================================
def tournament_selection(population, fitnesses, k=3):
    selected = []
    pop_size = len(population)
    for _ in range(pop_size):
        aspirants = random.sample(range(pop_size), k)
        best_idx = max(aspirants, key=lambda idx: fitnesses[idx])
        selected.append(list(population[best_idx]))
    return selected

def single_point_crossover(p1, p2, pc=0.8):
    if random.random() < pc and len(p1) > 1:
        point = random.randint(1, len(p1) - 1)
        c1 = p1[:point] + p2[point:]
        c2 = p2[:point] + p1[point:]
        return c1, c2
    return list(p1), list(p2)

def binary_mutation(individual, pm=0.05):
    for i in range(len(individual)):
        if random.random() < pm:
            individual[i] = 1 - individual[i]
    return individual

# ==========================================
# 4. MOTOR DO ALGORITMO GENÉTICO
# ==========================================
def run_genetic_algorithm(strategy_func, generations=50, pop_size=50):
    # Inicialização aleatória da população
    population = [[random.randint(0, 1) for _ in range(15)] for _ in range(pop_size)]

    history_mean = []
    history_std = []

    best_overall_ind = None
    best_overall_fit = -1.0

    for gen in range(generations):
        fitnesses = [strategy_func(ind) for ind in population]

        # Registra estatísticas da geração
        mean_fit = statistics.mean(fitnesses)
        std_fit = statistics.stdev(fitnesses) if pop_size > 1 else 0.0
        history_mean.append(mean_fit)
        history_std.append(std_fit)

        # Atualiza melhor global
        max_gen_fit = max(fitnesses)
        max_gen_idx = fitnesses.index(max_gen_fit)
        if max_gen_fit > best_overall_fit:
            best_overall_fit = max_gen_fit
            best_overall_ind = list(population[max_gen_idx])

        # Seleção, Crossover e Mutação
        selected = tournament_selection(population, fitnesses)

        next_population = []
        for i in range(0, pop_size, 2):
            p1 = selected[i]
            p2 = selected[(i + 1) % pop_size]
            c1, c2 = single_point_crossover(p1, p2)
            next_population.append(binary_mutation(c1))
            next_population.append(binary_mutation(c2))

        population = next_population[:pop_size]

    return best_overall_ind, best_overall_fit, history_mean, history_std

# ==========================================
# 5. EXECUÇÃO E COMPARAÇÃO DOS TESTES
# ==========================================
if __name__ == "__main__":
    random.seed(42) # Para reprodutibilidade

    print("Executando Estratégia A (Penalidade Rígida)...")
    best_a, fit_a, mean_a, std_a = run_genetic_algorithm(evaluate_strategy_a)

    random.seed(42)
    print("Executando Estratégia B (Penalidade Proporcional)...")
    best_b, fit_b, mean_b, std_b = run_genetic_algorithm(evaluate_strategy_b)

    print("\n" + "="*70)
    print(f"{'MÉTRICA / RESULTADO':<30} | {'ESTRATÉGIA A (Rígida)':<18} | {'ESTRATÉGIA B (Proporcional)'}")
    print("="*70)
    print(f"{'Melhor Fitness Encontrado':<30} | {fit_a:<18.2f} | {fit_b:.2f}")

    ram_a = sum(r * g for r, g in zip(RAM, best_a))
    cpu_a = sum(c * g for c, g in zip(CPU, best_a))
    ram_b = sum(r * g for r, g in zip(RAM, best_b))
    cpu_b = sum(c * g for c, g in zip(CPU, best_b))

    print(f"{'Uso de RAM (Max 16GB)':<30} | {ram_a:<18.1f} | {ram_b:.1f}")
    print(f"{'Uso de CPU (Max 8C)':<30} | {cpu_a:<18.1f} | {cpu_b:.1f}")
    print(f"{'Média Final do Fitness':<30} | {mean_a[-1]:<18.2f} | {mean_b[-1]:.2f}")
    print(f"{'Desvio-Padrão Final':<30} | {std_a[-1]:<18.2f} | {std_b[-1]:.2f}")
    print("="*70)
    print(f"Melhor Vetor (Estratégia A): {best_a}")
    print(f"Melhor Vetor (Estratégia B): {best_b}")

Executando Estratégia A (Penalidade Rígida)...
Executando Estratégia B (Penalidade Proporcional)...

MÉTRICA / RESULTADO            | ESTRATÉGIA A (Rígida) | ESTRATÉGIA B (Proporcional)
Melhor Fitness Encontrado      | 405.00             | 405.00
Uso de RAM (Max 16GB)          | 16.0               | 16.0
Uso de CPU (Max 8C)            | 7.9                | 7.9
Média Final do Fitness         | 306.00             | 369.76
Desvio-Padrão Final            | 139.49             | 36.03
Melhor Vetor (Estratégia A): [1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1]
Melhor Vetor (Estratégia B): [1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1]
